# SENTIMENT ANALYSIS OF TWEETS- MAJOR PROJECT

##Data Preparation

### Loading the Preprocessed Data

In [1]:
import zipfile
zip_ref = zipfile.ZipFile('archive.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [2]:
import pandas as pd

# Loading the preprocessed dataset
df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding='latin-1', header=None)

In [3]:
# Renaming columns as done in minor project
df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

In [4]:
# Convert target variable to categorical labels
df['target'] = df['target'].apply(lambda x: 'positive' if x == 4 else 'negative')

### Feature Extraction

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initializing the vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)

# Fit and transform the text data
X = vectorizer.fit_transform(df['text'])
y = df['target']


### Data Splitting

In [6]:
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Initial Model Creation

### Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression

# Initializing the model
model = LogisticRegression()

# Training the model
model.fit(X_train, y_train)


LogisticRegression()

### Model Evaluation

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Making predictions
y_pred = model.predict(X_test)

# Evaluating the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.7404875

Classification Report:
               precision    recall  f1-score   support

    negative       0.76      0.70      0.73    159494
    positive       0.72      0.78      0.75    160506

    accuracy                           0.74    320000
   macro avg       0.74      0.74      0.74    320000
weighted avg       0.74      0.74      0.74    320000


Confusion Matrix:
 [[112085  47409]
 [ 35635 124871]]


## Model Improvement

### Trying Different Models

#### Random Forest

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initializing the model with optimized parameters
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

# Training the model
X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, train_size=0.5, random_state=42)
rf_model.fit(X_train_small, y_train_small)

# Predicting and evaluating
y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nRandom Forest Classification Report:\n", classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.680696875

Random Forest Classification Report:
               precision    recall  f1-score   support

    negative       0.78      0.50      0.61    159494
    positive       0.63      0.86      0.73    160506

    accuracy                           0.68    320000
   macro avg       0.71      0.68      0.67    320000
weighted avg       0.71      0.68      0.67    320000



####  Support Vector Machine (SVM)

In [10]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, train_size=0.2, random_state=42)

# Initializing the model with optimized parameters
svm_model = SVC(kernel='linear', C=0.1, random_state=42)

# Training the model
svm_model.fit(X_train_small, y_train_small)

# Predicting and evaluating
y_pred_svm = svm_model.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nSVM Classification Report:\n", classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.737303125

SVM Classification Report:
               precision    recall  f1-score   support

    negative       0.77      0.68      0.72    159494
    positive       0.71      0.80      0.75    160506

    accuracy                           0.74    320000
   macro avg       0.74      0.74      0.74    320000
weighted avg       0.74      0.74      0.74    320000



#### XGBoost

In [11]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Encode the target labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Initialize the XGBoost model
xgb_model = XGBClassifier(random_state=42)

# Train the model
xgb_model.fit(X_train, y_train_encoded)

# Predict and evaluate
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test_encoded, y_pred_xgb))
print("\nXGBoost Classification Report:\n", classification_report(y_test_encoded, y_pred_xgb))


XGBoost Accuracy: 0.727559375

XGBoost Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.62      0.70    159494
           1       0.69      0.83      0.75    160506

    accuracy                           0.73    320000
   macro avg       0.74      0.73      0.72    320000
weighted avg       0.74      0.73      0.72    320000



### Hyperparameter Tuning

#### Tuning Random Forest

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Initializing the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Defining the parameter grid with a focused range to reduce computation
param_grid_rf = {
    'n_estimators': [100, 200],   # Number of trees in the forest
    'max_depth': [10, 20],        # Maximum depth of the tree
    'min_samples_split': [2, 5],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2]    # Minimum number of samples required to be at a leaf node
}

# Initializing RandomizedSearchCV
random_search_rf = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid_rf,
    n_iter=10,          # Number of parameter settings that are sampled
    cv=3,               # Using 3-fold cross-validation for quicker results
    verbose=2,
    n_jobs=-1,          # Using all available cores for parallel processing
    random_state=42     # Ensure reproducibility
)

# Fitting the model
random_search_rf.fit(X_train, y_train)

# Best parameters and best score
print("Best Parameters:", random_search_rf.best_params_)
print("Best Score:", random_search_rf.best_score_)


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 10}
Best Score: 0.693239843200919


####  Tuning SVM

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, train_size=0.1, random_state=42)

# Initializing the SVM model with a fixed kernel to reduce complexity
svm_model = SVC(kernel='linear', random_state=42)

# Defining a simplified parameter grid
param_grid_svm = {
    'C': [0.1, 1]
}

# Initializing GridSearchCV for exhaustive search over the small grid
grid_search_svm = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid_svm,
    cv=2,
    verbose=2,
    n_jobs=-1
)

# Fitting the model
grid_search_svm.fit(X_train_small, y_train_small)

# Best parameters and best score
print("Best Parameters:", grid_search_svm.best_params_)
print("Best Score:", grid_search_svm.best_score_)


Fitting 2 folds for each of 2 candidates, totalling 4 fits
Best Parameters: {'C': 0.1}
Best Score: 0.733203125


###  Cross-Validation

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder


X_small, _, y_small, _ = train_test_split(X, y, train_size=0.05, random_state=42)

# Encoding labels to numeric values
label_encoder = LabelEncoder()
y_small_encoded = label_encoder.fit_transform(y_small)

# Lowering the number of features using SelectKBest or PCA (reduce dimensionality)
X_small = SelectKBest(chi2, k=500).fit_transform(X_small, y_small_encoded)


# Initializing the models with simpler parameters
rf_model = RandomForestClassifier(random_state=42)

# Using LinearSVC for faster computation with linear kernel
svm_model = LinearSVC(random_state=42, max_iter=1000)

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# 4. Create a StratifiedKFold object for more efficient cross-validation
skf = StratifiedKFold(n_splits=2)

# Cross-validate Random Forest
cv_scores_rf = cross_val_score(rf_model, X_small, y_small_encoded, cv=skf, n_jobs=-1, verbose=1)
print("Random Forest CV Average Accuracy:", cv_scores_rf.mean())

# Cross-validate SVM with LinearSVC
cv_scores_svm = cross_val_score(svm_model, X_small, y_small_encoded, cv=skf, n_jobs=-1, verbose=1)
print("SVM (LinearSVC) CV Average Accuracy:", cv_scores_svm.mean())

# Cross-validate XGBoost
cv_scores_xgb = cross_val_score(xgb_model, X_small, y_small_encoded, cv=skf, n_jobs=-1, verbose=1)
print("XGBoost CV Average Accuracy:", cv_scores_xgb.mean())



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   2 | elapsed:  1.5min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.


Random Forest CV Average Accuracy: 0.69505


[Parallel(n_jobs=-1)]: Done   2 out of   2 | elapsed:    1.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.


SVM (LinearSVC) CV Average Accuracy: 0.7342500000000001
XGBoost CV Average Accuracy: 0.7155125


[Parallel(n_jobs=-1)]: Done   2 out of   2 | elapsed:    8.5s finished


#### Ensemble Methods

In [18]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(X, y, train_size=0.1, random_state=42)


lr_model = LogisticRegression(max_iter=100)
rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
svm_model = LinearSVC(random_state=42, max_iter=1000)
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_estimators=50)

# Initialize Voting Classifier
voting_model = VotingClassifier(estimators=[
    ('lr', lr_model),
    ('rf', rf_model),
    ('svm', svm_model),
    ('xgb', xgb_model)], voting='hard', n_jobs=-1)  # Parallel processing

# Train the ensemble model on the smaller dataset
voting_model.fit(X_train_small, y_train_small)

# Predict and evaluate on the smaller test set
y_pred_voting = voting_model.predict(X_test_small)
print("Voting Classifier Accuracy:", accuracy_score(y_test_small, y_pred_voting))
print("\nVoting Classifier Classification Report:\n", classification_report(y_test_small, y_pred_voting))


Voting Classifier Accuracy: 0.7393069444444444

Voting Classifier Classification Report:
               precision    recall  f1-score   support

    negative       0.75      0.73      0.74    719965
    positive       0.73      0.75      0.74    720035

    accuracy                           0.74   1440000
   macro avg       0.74      0.74      0.74   1440000
weighted avg       0.74      0.74      0.74   1440000

